In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve

In [2]:
df=pd.read_csv('S05-hw-dataset.csv')

In [3]:
df.head()

,client_id,age,income,years_employed,credit_score,debt_to_income,num_credit_cards,num_late_payments,has_mortgage,has_car_loan,savings_balance,checking_balance,region_risk_score,phone_calls_to_support_last_3m,active_loans,customer_tenure_years,default
0,1,25,94074,22,839,0.547339,1,7,0,0,26057,5229,0.080052,19,1,8,0
1,2,58,51884,26,565,0.290882,1,1,0,1,16221,11595,0.428311,15,0,7,0
2,3,53,48656,39,561,0.522340,1,13,0,0,55448,-2947,0.770883,15,4,5,0
3,4,42,81492,30,582,0.709123,2,10,1,1,35188,17727,0.357619,0,2,7,1
4,5,42,94713,8,642,0.793392,3,3,0,0,0,-404,0.414260,17,1,10,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   client_id                       3000 non-null   int64  
 1   age                             3000 non-null   int64  
 2   income                          3000 non-null   int64  
 3   years_employed                  3000 non-null   int64  
 4   credit_score                    3000 non-null   int64  
 5   debt_to_income                  3000 non-null   float64
 6   num_credit_cards                3000 non-null   int64  
 7   num_late_payments               3000 non-null   int64  
 8   has_mortgage                    3000 non-null   int64  
 9   has_car_loan                    3000 non-null   int64  
 10  savings_balance                 3000 non-null   int64  
 11  checking_balance                3000 non-null   int64  
 12  region_risk_score               30

In [5]:
df.describe()

,client_id,age,income,years_employed,credit_score,debt_to_income,num_credit_cards,num_late_payments,has_mortgage,has_car_loan,savings_balance,checking_balance,region_risk_score,phone_calls_to_support_last_3m,active_loans,customer_tenure_years,default
count,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,1500.500000,45.059667,69658.992000,19.577667,649.285333,0.284065,3.494667,6.869333,0.495000,0.501333,20607.256667,5559.684333,0.400175,9.524667,1.976333,6.968667,0.410333
std,866.169729,14.192883,24742.235182,11.381497,69.955852,0.161112,2.289917,4.291278,0.500058,0.500082,14035.209739,6306.032612,0.204529,5.779030,1.408700,4.349942,0.491976
min,1.000000,21.000000,15000.000000,0.000000,402.000000,0.006147,0.000000,0.000000,0.000000,0.000000,0.000000,-3000.000000,0.001148,0.000000,0.000000,0.000000,0.000000
25%,750.750000,33.000000,52641.750000,10.000000,604.000000,0.157796,1.000000,3.000000,0.000000,0.000000,9612.250000,341.500000,0.239208,5.000000,1.000000,3.000000,0.000000
50%,1500.500000,45.000000,69784.500000,20.000000,647.000000,0.261726,3.000000,7.000000,0.000000,1.000000,20021.000000,5114.500000,0.381992,10.000000,2.000000,7.000000,0.000000
75%,2250.250000,57.000000,85874.250000,29.000000,697.000000,0.388886,6.000000,10.000000,1.000000,1.000000,30101.250000,9906.250000,0.549213,15.000000,3.000000,11.000000,1.000000
max,3000.000000,69.000000,156351.000000,39.000000,850.000000,0.878343,7.000000,14.000000,1.000000,1.000000,75237.000000,29335.000000,0.961733,19.000000,4.000000,14.000000,1.000000


In [7]:
df.value_counts(normalize=True)

,,,,,,,,,,,,,,,,,proportion
client_id,age,income,years_employed,credit_score,debt_to_income,num_credit_cards,num_late_payments,has_mortgage,has_car_loan,savings_balance,checking_balance,region_risk_score,phone_calls_to_support_last_3m,active_loans,customer_tenure_years,default,
3000,53,75302,13,692,0.093865,2,7,0,0,29853,1259,0.693861,6,2,9,0,0.000333
1,25,94074,22,839,0.547339,1,7,0,0,26057,5229,0.080052,19,1,8,0,0.000333
2,58,51884,26,565,0.290882,1,1,0,1,16221,11595,0.428311,15,0,7,0,0.000333
3,53,48656,39,561,0.522340,1,13,0,0,55448,-2947,0.770883,15,4,5,0,0.000333
4,42,81492,30,582,0.709123,2,10,1,1,35188,17727,0.357619,0,2,7,1,0.000333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13,57,69664,3,539,0.319360,0,6,0,1,31063,6166,0.581045,4,3,4,1,0.000333
12,68,98126,12,648,0.187963,0,3,0,0,31947,-730,0.819837,7,4,3,0,0.000333
11,46,72737,20,632,0.134631,7,10,0,0,29784,5907,0.222156,4,1,12,0,0.000333


Всего 3000 клиентов, 15 признаков, нормально распределённый таргет

In [9]:
y = df['default'].copy()
X = df.drop(columns=['default', 'client_id']).copy()

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
model = DummyClassifier(strategy="most_frequent", random_state=42)
model.fit(X_train, y_train)

DummyClassifier(random_state=42, strategy='most_frequent')

In [12]:
y_pred_classes = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred_classes)
roc_auc = roc_auc_score(y_test, y_pred_proba[:, 1])

print(f"accuracy: {accuracy:.4f}")
print(f"roc_auc_score: {roc_auc:.4f}")

accuracy: 0.5900
roc_auc_score: 0.5000


In [13]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    ))
])

In [15]:
param_grid = {
    'logreg__C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],  # параметр регуляризации
    'logreg__penalty': ['l2'],
    'logreg__solver': ['lbfgs', 'liblinear'],  # алгоритм оптимизации
}

# Создаем GridSearchCV
grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,                     # 5-кратная кросс-валидация
    scoring='roc_auc',        # оптимизируем по ROC-AUC
    n_jobs=-1,                # используем все доступные ядра
    verbose=1                 # вывод информации о процессе
)

grid_search.fit(X_train, y_train)

Начинаем подбор гиперпараметров...
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Подбор завершен!


In [18]:
# Вычисляем ROC-кривую
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Создаем график
plt.figure(figsize=(10, 8))

# ROC-кривая нашей модели
plt.plot(fpr, tpr, color='darkorange', lw=3,
         label=f'Logistic Regression (AUC = {roc_auc:.4f})')

# Линия случайного классификатора
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--',
         label='Random Classifier (AUC = 0.5)')

# Настраиваем график
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=14)
plt.ylabel('True Positive Rate (TPR)', fontsize=14)
plt.title('ROC Curve - Logistic Regression', fontsize=16, fontweight='bold')
plt.legend(loc="lower right", fontsize=12)
plt.grid(True, alpha=0.3)

# Добавляем дополнительные элементы
plt.fill_between(fpr, tpr, alpha=0.1, color='darkorange')
plt.minorticks_on()
plt.grid(which='major', linestyle='-', linewidth='0.5', alpha=0.7)
plt.grid(which='minor', linestyle=':', linewidth='0.5', alpha=0.5)


plt.savefig("homeworks/HW05/figures", dpi=300, bbox_inches='tight')

plt.show()

ValueError: y should be a 1d array, got an array of shape (600, 2) instead.